In [21]:
import json
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../dadosDesafio/dados_nivel_1.json")

with DATA_PATH.open(encoding="utf-8") as file:
    dataset = json.load(file)

operacoes = pd.DataFrame(dataset["operacoes"])

taxa_usd_brl = dataset["taxa_cambio_usd_brl"]

print(f"Taxa USD/BRL: {taxa_usd_brl}")
print(f"Quantidade de operações: {len(operacoes)}")


operacoes.head()

operacoes.info()

operacoes.describe(include="all")

print("Valores ausentes:")
display(operacoes.isna().sum())

duplicados = operacoes[operacoes.duplicated(subset="id", keep=False)].sort_values("id")

display(duplicados)

operacoes = operacoes.drop_duplicates(subset="id", keep="first").copy()

operacoes["data"] = pd.to_datetime(
    operacoes["data"],
    errors="coerce"
)

print(f"Operações após deduplicação: {len(operacoes)}"),
print(f"IDs únicos: {operacoes['id'].nunique()}")
print(f"Datas ausentes: {operacoes['data'].isna().sum()}")


operacoes["valor"] = pd.to_numeric(
    operacoes["valor"],
    errors="coerce"
)


operacoes["valor_brl"] = operacoes["valor"].astype(float)


operacoes.loc[
    operacoes["moeda"] == "USD",
    "valor_brl"
] = (
    operacoes.loc[
        operacoes["moeda"] == "USD",
        "valor"
    ] * taxa_usd_brl
)

display(
    operacoes[
        ["id", "valor", "moeda", "valor_brl"]
    ]
)

volume_por_cliente = (
    operacoes.groupby("cliente_id", as_index=False)
      .agg(
          volume_total_brl=("valor_brl", "sum")
      )
      .sort_values(
          "volume_total_brl",
          ascending=False
      )
)

display(volume_por_cliente)

operacoes_por_canal = (
    operacoes.groupby("canal")
      .size()
      .reset_index(name="quantidade_operacoes")
      .sort_values(
          "quantidade_operacoes",
          ascending=False
      )
)

display(operacoes_por_canal)


Taxa USD/BRL: 5.4
Quantidade de operações: 20
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           20 non-null     str  
 1   cliente_id   20 non-null     str  
 2   data         19 non-null     str  
 3   valor        20 non-null     int64
 4   moeda        20 non-null     str  
 5   canal        20 non-null     str  
 6   tipo         20 non-null     str  
 7   contraparte  20 non-null     str  
 8   observacao   20 non-null     str  
dtypes: int64(1), str(8)
memory usage: 1.5 KB
Valores ausentes:


id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


Operações após deduplicação: 19
IDs únicos: 19
Datas ausentes: 1


,id,valor,moeda,valor_brl
0,OP-0001,18100,BRL,18100.0
1,OP-0002,17300,BRL,17300.0
2,OP-0003,18800,BRL,18800.0
3,OP-0004,3300,BRL,3300.0
4,OP-0005,25900,BRL,25900.0
5,OP-0006,27000,BRL,27000.0
6,OP-0007,17200,BRL,17200.0
7,OP-0008,15200,BRL,15200.0
8,OP-0009,16100,BRL,16100.0
10,OP-0010,3800,BRL,3800.0


,cliente_id,volume_total_brl
3,CLI-A-4,79500.0
0,CLI-A-1,57500.0
1,CLI-A-2,52900.0
2,CLI-A-3,48500.0
4,CLI-A-5,16900.0
5,CLI-A-6,10200.0


,canal,quantidade_operacoes
3,pix,8
4,ted,5
0,boleto,3
1,cartao,2
2,especie,1


# Desafio Itaú Estágio Engenharia de IA — Nível 1
## Prevenção à Lavagem de Dinheiro

Este notebook apresenta o tratamento dos dados, a aplicação de regras
determinísticas e a análise interpretativa de um cliente sinalizado
utilizando um modelo de linguagem.

A solução mantém separadas as responsabilidades:

- pandas: limpeza, agregações e cálculos determinísticos;
- regras: identificação objetiva dos sinais definidos no desafio;
- LLM: interpretação dos sinais e redação do parecer;
- validação estruturada: garantia de que a resposta da LLM segue o formato esperado.

## Tratamento dos problemas de qualidade

Foram identificados dois problemas relevantes:

1. O identificador `OP-0007` aparece duas vezes com os mesmos valores
   em todos os campos. A segunda ocorrência foi considerada uma duplicação
   do registro e apenas uma ocorrência foi mantida.

2. A operação `OP-0017` possui data ausente. A data não foi inferida ou
   preenchida artificialmente, pois não existe evidência suficiente para
   determinar o dia correto. Essa operação permanece na base para as
   análises que não dependem de data, mas não participa da regra de
   fracionamento, que exige operações realizadas na mesma data.

A deduplicação é importante porque uma duplicação alteraria contagens,
medianas e agregações. Já a preservação da operação sem data evita a
introdução de uma informação que não está presente na fonte.